# MobileNetV3-Small Parking Slot Occupancy: 3-Fold Leave-One-Lot-Out (LOO) GPU Training

This notebook trains and evaluates the MobileNetV3-small slot classifier and evaluates YOLO11n baseline on Google Colab (GPU).

### Cross-Validation Protocol:
- **Leave-One-Lot-Out (LOO)** across all 3 PKLot facilities:
  * **Fold 0**: Test on `PUCPR`, Train/Val on `UFPR04` + `UFPR05`
  * **Fold 1**: Test on `UFPR04`, Train/Val on `PUCPR` + `UFPR05`
  * **Fold 2**: Test on `UFPR05`, Train/Val on `PUCPR` + `UFPR04`
- **Data Leakage Guarantee**: Validation split is partitioned strictly by day inside training lots; zero frame overlap across splits (`assert_no_data_leakage`).
- **Calibration**: Computes Expected Calibration Error (ECE), reliability diagrams, and validation-fitted temperature scaling.

## 1. Environment Setup & GPU Check

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")

## 2. Clone Repository & Install Dependencies

In [ ]:
%cd /content
!if [ ! -d "smart-parking-platform" ]; then git clone -b vision-fusion https://github.com/bh00mi-11/smart-parking-platform.git; fi
%cd /content/smart-parking-platform/vision
!pip install -r requirements.txt

## 3. Download & Prepare Datasets
Downloads full PKLot dataset (PUCPR, UFPR04, UFPR05 across Sunny, Cloudy, and Rainy weather days).

In [ ]:
!python scripts/download_data.py
!python scripts/inspect_data.py

## 4. Run 3-Fold Leave-One-Lot-Out (LOO) Training
Runs full GPU training with Cosine/AdamW schedule, class frequency loss weighting, and validation temperature scaling across all 3 folds.

In [ ]:
for fold in [0, 1, 2]:
    print(f"\n{'='*60}\nSTARTING LOO FOLD {fold}\n{'='*60}")
    !python src/train_classifier.py --epochs 8 --batch-size 64 --lr 3e-4 --protocol loo --fold {fold}

## 5. Aggregate LOO Metrics & Print Results Table

In [ ]:
import json
from pathlib import Path
import pandas as pd

lots = ["PUCPR", "UFPR04", "UFPR05"]
rows = []

for fold, lot in enumerate(lots):
    metrics_file = Path(f"outputs/loo_fold{fold}_{lot}_metrics.json")
    if metrics_file.exists():
        with open(metrics_file) as f:
            m = json.load(f)
        rows.append({
            "Fold": fold,
            "Test Lot": lot,
            "Accuracy": f"{m['accuracy']*100:.2f}%",
            "Precision": f"{m['precision']:.4f}",
            "Recall": f"{m['recall']:.4f}",
            "F1 Score": f"{m['f1_score']:.4f}",
            "ECE (Uncal)": f"{m['ece_uncalibrated']:.4f}",
            "ECE (Cal)": f"{m['ece_calibrated']:.4f}",
            "Latency (ms/crop)": f"{m['inference_latency_cpu_ms_per_crop']:.2f}"
        })

df_results = pd.DataFrame(rows)
print("\n=== LEAVE-ONE-LOT-OUT (LOO) EVALUATION SUMMARY ===")
print(df_results.to_markdown(index=False))

## 6. Run Per-Lot YOLO11n Vehicle Counting Comparison

In [ ]:
!python src/eval_yolo_lots.py

## 7. Save Weights, Metrics, and Diagrams to Google Drive

In [ ]:
import shutil
drive_save_dir = "/content/drive/MyDrive/smart_parking_vision_loo_outputs"
os.makedirs(drive_save_dir, exist_ok=True)

for f in Path("outputs").glob("*"):
    if f.is_file():
        dest = os.path.join(drive_save_dir, f.name)
        shutil.copy(str(f), dest)
        print(f"Backed up: {f.name} -> {drive_save_dir}")
print("All training artifacts successfully saved to Google Drive!")